In [1]:
import tensorflow as tf
import sklearn
from keras.applications import EfficientNetV2B0, ResNet50, VGG16, MobileNetV2, Xception, MobileNetV3Small
from keras.optimizers import Adam
from keras.preprocessing import image_dataset_from_directory
from keras.losses import BinaryCrossentropy
from keras.callbacks import EarlyStopping
import kagglehub
import os
from datasets import load_dataset


c:\Users\donof\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# ds = load_dataset("ILSVRC/imagenet-1k")
pretrained_imagenet = load_dataset("timm/mini-imagenet")

In [2]:
LEARNING_RATE=0.0001
BATCH_SIZE=64
IMAGE_SIZE=160
VAL_SPLIT=0.2

In [3]:
efficientnetv2b0 = EfficientNetV2B0(weights='imagenet', input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3), include_top=False)
resnet50 = ResNet50(weights='imagenet', input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3), include_top=False)
vgg16 = VGG16(weights='imagenet', input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3), include_top=False)
xception = Xception(weights='imagenet', input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3), include_top=False)
mobilenetv3small = MobileNetV3Small(weights='imagenet', input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3), include_top=False)
mobilenetv2 = MobileNetV2(weights='imagenet', input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3), include_top=False)

In [4]:
dataset_path = kagglehub.dataset_download("doctorstrange420/real-and-fake-ai-generated-art-images-dataset")
dataset_path

'C:\\Users\\donof\\.cache\\kagglehub\\datasets\\doctorstrange420\\real-and-fake-ai-generated-art-images-dataset\\versions\\1'

In [ ]:
os.listdir(dataset_path+"\Data")

In [5]:
fake_dir = os.path.join(dataset_path+"\Data", "FAKE")
fake_dir

'C:\\Users\\donof\\.cache\\kagglehub\\datasets\\doctorstrange420\\real-and-fake-ai-generated-art-images-dataset\\versions\\1\\Data\\FAKE'

In [6]:
real_dir = os.path.join(dataset_path+"/Data", "REAL")
real_dir

'C:\\Users\\donof\\.cache\\kagglehub\\datasets\\doctorstrange420\\real-and-fake-ai-generated-art-images-dataset\\versions\\1/Data\\REAL'

In [ ]:
os.listdir(real_dir)[:5]

In [ ]:
os.listdir(fake_dir)[:5]

In [9]:
trainingDs = image_dataset_from_directory(dataset_path+"\Data", seed=123, subset="training", validation_split=VAL_SPLIT, batch_size=BATCH_SIZE, image_size=(IMAGE_SIZE, IMAGE_SIZE))
trainingDs

Found 21642 files belonging to 2 classes.
Using 17314 files for training.


<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 160, 160, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int32, name=None))>

In [8]:
valDs = image_dataset_from_directory(dataset_path+"\Data", seed=123, subset="validation", validation_split=VAL_SPLIT, batch_size=BATCH_SIZE, image_size=(IMAGE_SIZE, IMAGE_SIZE))
valDs

Found 21642 files belonging to 2 classes.
Using 4328 files for validation.


<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 160, 160, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int32, name=None))>

In [7]:
data_augmentation = tf.keras.Sequential(
    [
        tf.keras.layers.RandomFlip("horizontal"),
        tf.keras.layers.RandomRotation(0.1),
        tf.keras.layers.RandomZoom(0.1),
        tf.keras.layers.RandomBrightness(0.1)
    ]
)
data_augmentation

In [10]:
earlyStopping = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

In [ ]:
vgg16.trainable=True

for layer in vgg16.layers[:-30]:
    layer.trainable = False

In [ ]:
# x = data_augmentation(tf.keras.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3)))
# x = tf.keras.applications.vgg16.preprocess_input(x)
x = vgg16.output
# x = data_augmentation(x)
# x = vgg16(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)#GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dense(128, activation='relu')(x)
x = tf.keras.layers.Dropout(0.5)(x)
output = tf.keras.layers.Dense(1, activation='sigmoid')(x)

In [ ]:
# For VGG16 (and apply same pattern to other models)


In [ ]:
vgg16.summary()

In [ ]:
efficientnetv2b0.trainable=True

for layer in efficientnetv2b0.layers[:-30]:
    layer.trainable = False

In [ ]:
# x = data_augmentation(tf.keras.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3)))
# x = tf.keras.applications.efficientnet_v2.preprocess_input(x)
x = efficientnetv2b0.output
# x = efficientnetv2b0(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dense(128, activation='relu')(x)
x = tf.keras.layers.Dropout(0.5)(x)
output = tf.keras.layers.Dense(1, activation='sigmoid')(x)

In [ ]:
efficientnetv2b0 = tf.keras.Model(inputs=efficientnetv2b0.input, outputs=output)

In [ ]:
efficientnetv2b0.summary()

In [ ]:
efficientnetv2b0.compile(optimizer=Adam(learning_rate=LEARNING_RATE), loss=BinaryCrossentropy(), 
              metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()])

In [19]:
resnet50.trainable=True

for layer in resnet50.layers[:-30]:
    layer.trainable = False

In [ ]:
# # x = data_augmentation(tf.keras.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3)))
# # x = tf.keras.applications.resnet50.preprocess_input(x)
# x = resnet50.output
# # x = resnet50(x, training=False)
# x = tf.keras.layers.GlobalAveragePooling2D()(x)
# x = tf.keras.layers.Dense(128, activation='relu')(x)
# x = tf.keras.layers.Dropout(0.5)(x)
# output = tf.keras.layers.Dense(1, activation='sigmoid')(x)

In [20]:
inputs = tf.keras.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3))
x = data_augmentation(inputs)  # Add augmentation here
x = tf.keras.applications.resnet50.preprocess_input(x)
x = resnet50(x, training=False)  # Use the pretrained base model
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dense(128, activation='relu')(x)
x = tf.keras.layers.Dropout(0.5)(x)
output = tf.keras.layers.Dense(1, activation='sigmoid')(x)

resnet50= tf.keras.Model(inputs=inputs, outputs=output)
resnet50.compile(optimizer=Adam(learning_rate=LEARNING_RATE), 
                    loss=BinaryCrossentropy(), 
                    metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()])

In [ ]:
# resnet50 = tf.keras.Model(inputs=resnet50.inputs, outputs=output)

In [21]:
resnet50.summary()

Model: "model_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_9 (InputLayer)        [(None, 160, 160, 3)]     0         
                                                                 
 sequential (Sequential)     (None, 160, 160, 3)       0         
                                                                 
 tf.__operators__.getitem (  (None, 160, 160, 3)       0         
 SlicingOpLambda)                                                
                                                                 
 tf.nn.bias_add (TFOpLambda  (None, 160, 160, 3)       0         
 )                                                               
                                                                 
 resnet50 (Functional)       (None, 5, 5, 2048)        23587712  
                                                                 
 global_average_pooling2d_2  (None, 2048)              0   

In [ ]:
# resnet50.compile(optimizer=Adam(learning_rate=LEARNING_RATE), loss=BinaryCrossentropy(), 
#               metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()])

In [15]:
mobilenetv2.trainable=True

for layer in mobilenetv2.layers[:-30]:
    layer.trainable = False

In [ ]:
# # x = data_augmentation(tf.keras.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3)))
# # x = tf.keras.applications.resnet50.preprocess_input(x)
# x = mobilenetv2.output
# # x = resnet50(x, training=False)
# x = tf.keras.layers.GlobalAveragePooling2D()(x)
# x = tf.keras.layers.Dense(128, activation='relu')(x)
# x = tf.keras.layers.Dropout(0.5)(x)
# output = tf.keras.layers.Dense(1, activation='sigmoid')(x)

In [16]:
inputs = tf.keras.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3))
x = data_augmentation(inputs)  # Add augmentation here
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = mobilenetv2(x, training=False)  # Use the pretrained base model
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dense(128, activation='relu')(x)
x = tf.keras.layers.Dropout(0.5)(x)
output = tf.keras.layers.Dense(1, activation='sigmoid')(x)

mobilenetv2= tf.keras.Model(inputs=inputs, outputs=output)
mobilenetv2.compile(optimizer=Adam(learning_rate=LEARNING_RATE), 
                    loss=BinaryCrossentropy(), 
                    metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()])

In [ ]:
# mobilenetv2 = tf.keras.Model(inputs=mobilenetv2.input, outputs=output)

In [17]:
mobilenetv2.summary()

Model: "model_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_8 (InputLayer)        [(None, 160, 160, 3)]     0         
                                                                 
 sequential (Sequential)     (None, 160, 160, 3)       0         
                                                                 
 tf.math.truediv (TFOpLambd  (None, 160, 160, 3)       0         
 a)                                                              
                                                                 
 tf.math.subtract (TFOpLamb  (None, 160, 160, 3)       0         
 da)                                                             
                                                                 
 mobilenetv2_1.00_160 (Func  (None, 5, 5, 1280)        2257984   
 tional)                                                         
                                                           

In [ ]:
# mobilenetv2.compile(optimizer=Adam(learning_rate=LEARNING_RATE), loss=BinaryCrossentropy(), 
#               metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()])

In [ ]:
xception.trainable=True

for layer in xception.layers[:-30]:
    layer.trainable = False

In [ ]:
# x = data_augmentation(tf.keras.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3)))
# x = tf.keras.applications.resnet50.preprocess_input(x)
x = xception.output
# x = resnet50(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dense(128, activation='relu')(x)
x = tf.keras.layers.Dropout(0.5)(x)
output = tf.keras.layers.Dense(1, activation='sigmoid')(x)

In [ ]:
xception = tf.keras.Model(inputs=xception.input, outputs=output)

In [ ]:
xception.summary()

In [ ]:
xception.compile(optimizer=Adam(learning_rate=LEARNING_RATE), loss=BinaryCrossentropy(), 
              metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()])

In [11]:
mobilenetv3small.trainable=True

for layer in mobilenetv3small.layers[:-30]:
    layer.trainable = False

In [ ]:
# x = data_augmentation(tf.keras.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3)))
# # x = tf.keras.applications.resnet50.preprocess_input(x)
# x = mobilenetv3small.output
# # x = resnet50(x, training=False)
# x = tf.keras.layers.GlobalAveragePooling2D()(x)
# x = tf.keras.layers.Dense(128, activation='relu')(x)
# x = tf.keras.layers.Dropout(0.5)(x)
# output = tf.keras.layers.Dense(1, activation='sigmoid')(x)

In [12]:
inputs = tf.keras.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3))
x = data_augmentation(inputs)  # Add augmentation here
# x = tf.keras.applications.mobilenet_v3.preprocess_input(x)
x = mobilenetv3small(x, training=False)  # Use the pretrained base model
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dense(128, activation='relu')(x)
x = tf.keras.layers.Dropout(0.5)(x)
output = tf.keras.layers.Dense(1, activation='sigmoid')(x)

mobilenetv3small= tf.keras.Model(inputs=inputs, outputs=output)
mobilenetv3small.compile(optimizer=Adam(learning_rate=LEARNING_RATE), 
                    loss=BinaryCrossentropy(), 
                    metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()])

In [ ]:
# mobilenetv3small = tf.keras.Model(inputs=mobilenetv3small.input, outputs=output)

In [13]:
mobilenetv3small.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_7 (InputLayer)        [(None, 160, 160, 3)]     0         
                                                                 
 sequential (Sequential)     (None, 160, 160, 3)       0         
                                                                 
 MobilenetV3small (Function  (None, 5, 5, 576)         939120    
 al)                                                             
                                                                 
 global_average_pooling2d (  (None, 576)               0         
 GlobalAveragePooling2D)                                         
                                                                 
 dense (Dense)               (None, 128)               73856     
                                                                 
 dropout (Dropout)           (None, 128)               0     

In [ ]:
# mobilenetv3small.compile(optimizer=Adam(learning_rate=LEARNING_RATE), loss=BinaryCrossentropy(), 
#               metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()])

In [18]:
history = mobilenetv2.fit(
    trainingDs,
    validation_data=valDs,
    # training_steps=trainingDs.samples//trainingDs.batch_size,
    # validation_steps=valDs.samples//valDs.batch_size,
    epochs=5,
    callbacks=[earlyStopping]
)

Epoch 1/5
271/271 [==============================] - 178s 622ms/step - loss: 0.3359 - accuracy: 0.8542 - precision_1: 0.8568 - recall_1: 0.8487 - val_loss: 0.4611 - val_accuracy: 0.7932 - val_precision_1: 0.7123 - val_recall_1: 0.9977
Epoch 2/5
271/271 [==============================] - 171s 629ms/step - loss: 0.2078 - accuracy: 0.9153 - precision_1: 0.9158 - recall_1: 0.9139 - val_loss: 0.6924 - val_accuracy: 0.7495 - val_precision_1: 0.6708 - val_recall_1: 0.9995
Epoch 3/5
271/271 [==============================] - 171s 629ms/step - loss: 0.1673 - accuracy: 0.9346 - precision_1: 0.9354 - recall_1: 0.9329 - val_loss: 0.6667 - val_accuracy: 0.7525 - val_precision_1: 0.6735 - val_recall_1: 0.9995


In [ ]:
history = efficientnetv2b0.fit(
    trainingDs,
    validation_data=valDs,
    # training_steps=trainingDs.samples//trainingDs.batch_size,
    # validation_steps=valDs.samples//valDs.batch_size,
    epochs=5,
    callbacks=[earlyStopping]
)

In [14]:
history = mobilenetv3small.fit(
    trainingDs,
    validation_data=valDs,
    # training_steps=trainingDs.samples//trainingDs.batch_size,
    # validation_steps=valDs.samples//valDs.batch_size,
    epochs=8,
    callbacks=[earlyStopping]
)

Epoch 1/8



271/271 [==============================] - 83s 274ms/step - loss: 0.5462 - accuracy: 0.7102 - precision: 0.7058 - recall: 0.7157 - val_loss: 0.3902 - val_accuracy: 0.8286 - val_precision: 0.8341 - val_recall: 0.8288
Epoch 2/8
271/271 [==============================] - 70s 257ms/step - loss: 0.3603 - accuracy: 0.8409 - precision: 0.8416 - recall: 0.8379 - val_loss: 0.2879 - val_accuracy: 0.8819 - val_precision: 0.8678 - val_recall: 0.9067
Epoch 3/8
271/271 [==============================] - 72s 264ms/step - loss: 0.2879 - accuracy: 0.8784 - precision: 0.8800 - recall: 0.8750 - val_loss: 0.2832 - val_accuracy: 0.8840 - val_precision: 0.8364 - val_recall: 0.9606
Epoch 4/8
271/271 [==============================] - 72s 264ms/step - loss: 0.2491 - accuracy: 0.8995 - precision: 0.8988 - recall: 0.8992 - val_loss: 0.2911 - val_accuracy: 0.8822 - val_precision: 0.8243 - val_recall: 0.9774
Epoch 5/8
271/271 [==============================] - 69s 255ms/step - loss: 0.2247 - accuracy: 0.9102 - pr

In [ ]:
history = xception.fit(
    trainingDs,
    validation_data=valDs,
    # training_steps=trainingDs.samples//trainingDs.batch_size,
    # validation_steps=valDs.samples//valDs.batch_size,
    epochs=5,
    callbacks=[earlyStopping]
)

In [ ]:
history = resnet50.fit(
    trainingDs,
    validation_data=valDs,
    # training_steps=trainingDs.samples//trainingDs.batch_size,
    # validation_steps=valDs.samples//valDs.batch_size,
    epochs=3,
    callbacks=[earlyStopping]
)

Epoch 1/3


In [ ]:
history = vgg16.fit(
    trainingDs,
    validation_data=valDs,
    # training_steps=trainingDs.samples//trainingDs.batch_size,
    # validation_steps=valDs.samples//valDs.batch_size,
    epochs=2,
    callbacks=[earlyStopping]
)